# SmartRec — Exploratory Data Analysis

This notebook delegates all computation to `data/eda.py`.  
Run `python data/processing.py` first to generate the processed parquet files.

In [1]:
import sys
from pathlib import Path

# Add project root to path so `data.eda` is importable
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

from data.eda import (
    load_processed_data,
    analyze_rating_distribution,
    analyze_user_activity,
    analyze_product_activity,
    analyze_temporal_trends,
    analyze_sparsity,
    analyze_filter_impact,
    analyze_categories,
    analyze_rating_activity_correlation,
    save_eda_summary,
    PROCESSED_DIR,
    RAW_DIR,
)

print('Imports OK')

Imports OK


## 1. Load processed data

Carrega os três parquets gerados pelo pipeline de pré-processamento (`data/processing.py`): **interactions** (user_id, product_id, rating, timestamp), **products** (metadados dos itens: título, descrição, categoria) e **users** (estatísticas agregadas por usuário). São a base para todas as análises e para os modelos de recomendação (CF e semântico).

In [2]:
interactions, products, users = load_processed_data(PROCESSED_DIR)
print(f'interactions : {len(interactions):,} rows')
print(f'products     : {len(products):,} rows')
print(f'users        : {len(users):,} rows')
interactions.head()

INFO: Loaded 770499 interactions, 9321 products, 98562 users


interactions : 770,499 rows
products     : 9,321 rows
users        : 98,562 rows


,user_id,product_id,rating,timestamp
0,A3J3BRHTDRFJ2G,0511189877,2.0,1397433600
1,A2OSUEZJIN7BI,0511189877,2.0,1478822400
2,A2I2KPNJDQ9SL0,0511189877,5.0,1472083200
3,A12JHGROAX49G7,0511189877,4.0,1457568000
4,A1D4UFZ9X6U5UW,0511189877,5.0,1437523200


## 2. Rating distribution

Analisa a distribuição das notas (1–5) nas interações. Mostra média, mediana, desvio padrão e percentuais por nota. Ajuda a identificar viés de avaliação (ex.: predominância de 5 estrelas) e impacta a calibração dos modelos colaborativos (SVD) e métricas como NDCG.

In [3]:
rating_stats = analyze_rating_distribution(interactions)
rating_stats

INFO: generated new fontManager
INFO: Saved C:\Users\ander\projetos\smartrec\reports\figures\rating_distribution.png


{'mean': 4.335522651672363,
 'median': 5.0,
 'std': 1.1179829835891724,
 'min': 1.0,
 'max': 5.0,
 'pct_1': 0.05331479988942231,
 'pct_2': 0.03968986332234046,
 'pct_3': 0.07451015510727464,
 'pct_4': 0.18312807673987896,
 'pct_5': 0.6493571049410837}

## 3. User activity distribution

Quantifica quantas interações cada usuário tem (reviews por usuário). A mediana e percentis (p25, p75, p95) indicam se o dataset é equilibrado ou dominado por power users. Essencial para entender cold start — usuários com poucas interações exigem fallback semântico no modelo híbrido.

In [4]:
user_stats = analyze_user_activity(interactions)
user_stats

INFO: Saved C:\Users\ander\projetos\smartrec\reports\figures\user_activity_distribution.png


{'mean': 7.817404273452243,
 'median': 6.0,
 'std': 5.15054440978261,
 'min': 5.0,
 'max': 213.0,
 'p25': 5.0,
 'p75': 8.0,
 'p95': 16.0,
 'p99': 28.0}

## 4. Product activity distribution

Mede quantas avaliações cada produto recebeu. A distribuição costuma ser long-tail: poucos itens populares e muitos com poucas interações. Afeta a qualidade das recomendações CF (produtos cold-start exigem busca semântica) e orienta thresholds de inclusão no pipeline.

In [5]:
product_stats = analyze_product_activity(interactions)
product_stats

INFO: Saved C:\Users\ander\projetos\smartrec\reports\figures\product_activity_distribution.png


{'mean': 51.79477009948911,
 'median': 23.0,
 'std': 116.28121763782565,
 'min': 10.0,
 'max': 3487.0,
 'p25': 14.0,
 'p75': 47.0,
 'p95': 175.0,
 'p99': 479.25}

## 5. Temporal trends

Analisa a evolução das interações ao longo do tempo (mensal). Revela picos de atividade, sazonalidade e janela temporal dos dados. Útil para validação temporal, split treino/teste cronológico e decisões sobre decay temporal nas recomendações.

In [6]:
temporal_stats = analyze_temporal_trends(interactions)
temporal_stats

INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO: Saved C:\Users\ander\projetos\smartrec\reports\figures\temporal_trends.png


{'min_date': '1999-07',
 'max_date': '2018-06',
 'n_months': 225,
 'peak_month': '2013-01',
 'peak_count': 15537}

## 6. Sparsity

Mede a esparsidade da matriz usuário × produto: a fração de células vazias. Alta sparsity (quase 100%) é típica em sistemas de recomendação e desafia modelos colaborativos (SVD, KNN). A heatmap amostra uma submatriz para visualizar o padrão de dados ausentes — justifica a fusão com busca semântica no modelo híbrido.

In [8]:
!pip install seaborn



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
sparsity_stats = analyze_sparsity(interactions)
sparsity_stats

INFO: Saved C:\Users\ander\projetos\smartrec\reports\figures\sparsity_heatmap.png


{'n_users': 98562.0,
 'n_products': 14876.0,
 'n_interactions': 770499.0,
 'sparsity': 0.9994744955449414,
 'density': 0.0005255044550586342,
 'sample_sparsity': 0.9372}

## 7. Filter impact

Compara o volume de reviews bruto (`data/raw/reviews.csv`) com o processado após filtros (ex.: mín. 5 interações por usuário/produto). Mostra quantos dados foram retidos ou descartados e valida a configuração do pipeline de pré-processamento — importante para documentar o impacto dos critérios de qualidade.

In [10]:
filter_stats = analyze_filter_impact(RAW_DIR, interactions)
filter_stats

INFO: Saved C:\Users\ander\projetos\smartrec\reports\figures\filter_impact.png


{'n_raw': 0, 'n_processed': 770499, 'pct_retained': 0.0, 'n_dropped': 0}

## 8. Category distribution

Distribuição de produtos e interações por categoria (ex.: Electronics, Camera, Computers). Revela categorias dominantes e nichos, auxilia na interpretação das recomendações semânticas (embedding por título/descrição) e ajuda a detectar desbalanceamento no catálogo.

In [11]:
cat_stats = analyze_categories(products, interactions)
cat_stats

INFO: Saved C:\Users\ander\projetos\smartrec\reports\figures\category_distribution.png


{'n_categories': 27,
 'top_categories': [{'category': 'All Electronics',
   'n_products': 3045,
   'n_interactions': 132553},
  {'category': 'Home Audio & Theater',
   'n_products': 1797,
   'n_interactions': 76207},
  {'category': 'Camera & Photo', 'n_products': 1929, 'n_interactions': 74618},
  {'category': 'Computers', 'n_products': 1310, 'n_interactions': 54413},
  {'category': 'Cell Phones & Accessories',
   'n_products': 476,
   'n_interactions': 18741},
  {'category': 'Office Products', 'n_products': 66, 'n_interactions': 3836},
  {'category': 'Car Electronics', 'n_products': 154, 'n_interactions': 3810},
  {'category': 'Musical Instruments',
   'n_products': 56,
   'n_interactions': 3081},
  {'category': 'Industrial & Scientific',
   'n_products': 36,
   'n_interactions': 3072},
  {'category': 'Tools & Home Improvement',
   'n_products': 85,
   'n_interactions': 2818},
  {'category': 'Amazon Devices', 'n_products': 37, 'n_interactions': 1704},
  {'category': 'Automotive', 'n_pr

## 9. Rating × activity correlation

Verifica se usuários mais ativos tendem a dar notas diferentes dos ocasionais (Pearson e Spearman entre quantidade de reviews e nota média). Correlação próxima de zero sugere ausência de viés de seleção nas notas — dados mais homogêneos para treino do modelo colaborativo.

In [12]:
corr_stats = analyze_rating_activity_correlation(interactions, users)
corr_stats

INFO: Saved C:\Users\ander\projetos\smartrec\reports\figures\rating_activity_correlation.png


{'pearson_r': 0.026617392954777077, 'spearman_r': -0.009573703448239938}

## 10. Export summary

Consolida todas as métricas das seções anteriores em um único JSON (`reports/eda_summary.json`). Serve como registro versionável do estado dos dados, documentação e base para scripts de treino, configuração de hiperparâmetros ou dashboards automatizados.

In [13]:
summary = {
    'rating_distribution': rating_stats,
    'user_activity': user_stats,
    'product_activity': product_stats,
    'temporal_trends': temporal_stats,
    'sparsity': sparsity_stats,
    'filter_impact': filter_stats,
    'categories': cat_stats,
    'rating_activity_correlation': corr_stats,
}
save_eda_summary(summary)
print('Summary saved to reports/eda_summary.json')

INFO: EDA summary saved to C:\Users\ander\projetos\smartrec\reports\eda_summary.json


Summary saved to reports/eda_summary.json
